# 08 · 神经网络抽象：Module / Linear / Sequential

> **本节属于 Part 4 · 神经网络抽象 nn。**

Part 3 我们已经能用 `Tensor` 端到端训练网络了，但训练循环里那些"收集参数、梯度清零、逐个更新"的样板代码很啰嗦。本节我们仿照 PyTorch，把这些封装成优雅的 **`Module / Linear / Sequential`**——从此搭网络像搭积木。

## 学习目标

- 理解 `Parameter` 与 `Module` 抽象：**自动收集参数、统一前向接口**
- 实现 `Linear`（全连接层）与 `Sequential`（层的串联）
- 用 `parameters()` 递归收集参数，用 `model(x)` 优雅前向
- 与 PyTorch 的 `nn.Module / nn.Linear / nn.Sequential` 一一对照

## 直觉与原理

PyTorch 的设计哲学很简单：

- **`Parameter`**：一个"会被记住的"张量，代表可学习的权重。
- **`Module`**：所有层和模型的基类。它会自动扫描自己的属性，把其中的 `Parameter` 和子 `Module` 都收集起来——于是 `model.parameters()` 一次拿到全部权重，`model.zero_grad()` 一次清零所有梯度。
- 调用 `model(x)` 实际上调用 `model.forward(x)`。

我们的 minitorch 完全采用同样的设计。先看 `Module` 是怎么递归收集参数的：

In [ ]:
import inspect
import numpy as np
import minitorch
from minitorch import Tensor, nn
from minitorch.utils import numerical_gradient, rel_error

print(inspect.getsource(nn.Module.parameters))

## Linear：全连接层

`Linear(in, out)` 持有一个权重 `W:(in,out)` 和偏置 `b:(out,)`，前向就是 `x @ W + b`。

In [ ]:
print(inspect.getsource(nn.Linear))

In [ ]:
lin = nn.Linear(3, 2)
x = Tensor(np.random.randn(4, 3))
out = lin(x)
print("输入形状:", x.shape, "-> 输出形状:", out.shape)
print("参数个数:", len(lin.parameters()), " （weight + bias）")
print("weight 形状:", lin.weight.shape, " bias 形状:", lin.bias.shape)

## Sequential：把层串起来

`Sequential` 把若干层依次连接，前一层输出喂给后一层。搭一个 MLP 就这么简单：

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
print("模型:", "784 -> [Linear+ReLU] -> 128 -> [Linear] -> 10")
print("参数张量个数:", len(model.parameters()))
total = sum(p.data.size for p in model.parameters())
print("可学习参数总量:", f"{total:,}")

## 验证：Linear 的前向与反向正确吗

用数值梯度检查权重的梯度。

In [ ]:
lin = nn.Linear(3, 2)
x_np = np.random.randn(5, 3)
out = lin(Tensor(x_np))
out.sum().backward()

def loss_np(Wv):
    return (x_np @ Wv + lin.bias.data).sum()

g = numerical_gradient(loss_np, lin.weight.data.copy())
print("weight 梯度相对误差:", rel_error(lin.weight.grad, g))

## PyTorch 对照

minitorch 的 nn API 刻意贴近 PyTorch：

| minitorch | PyTorch |
|---|---|
| `nn.Module` | `torch.nn.Module` |
| `nn.Linear(in, out)` | `nn.Linear(in, out)` |
| `nn.Sequential(...)` | `nn.Sequential(...)` |
| `model.parameters()` | `model.parameters()` |
| `model.zero_grad()` | `model.zero_grad()` |
| `model(x)` | `model(x)` |

一个差别：我们的 `Linear` 权重形状是 `(in, out)`、前向 `x @ W`；PyTorch 是 `(out, in)`、前向 `x @ W.T`。数学等价。把两边权重对齐后，前向结果应完全一致：

In [ ]:
import torch
import torch.nn as tnn

lin = nn.Linear(3, 2)
lt = tnn.Linear(3, 2).double()                         # minitorch 用 float64，这里也用 double
with torch.no_grad():
    lt.weight.copy_(torch.tensor(lin.weight.data.T))   # 注意转置（约定不同）
    lt.bias.copy_(torch.tensor(lin.bias.data))

x_np = np.random.randn(5, 3)
ours = lin(Tensor(x_np)).data
theirs = lt(torch.tensor(x_np)).detach().numpy()
print("前向结果一致:", np.allclose(ours, theirs))

## 📦 沉淀进 minitorch

`Parameter / Module / Sequential / Linear` 都在 **`minitorch/nn/`** 中，由 `tests/test_nn.py` 守护。后续所有网络都用它们搭建。

## 小练习

1. **手写一个自定义 Module**：继承 `nn.Module`，在 `__init__` 里放两个 `Linear`，在 `forward` 里实现 `relu(linear1(x))` 再 `linear2`。确认 `parameters()` 能收集到全部 4 个参数张量。
2. **无偏置层**：`nn.Linear(3, 2, bias=False)` 的参数个数是多少？前向还正确吗？
3. **思考**：`Module.parameters()` 为什么要递归处理子 `Module` 和列表？（提示：`Sequential` 的层就放在一个列表里。）

## 小结 & 下一站

✅ 我们造出了 `Module / Parameter / Linear / Sequential`——搭网络从此声明式、像搭积木。

**下一站 → `09_activations_and_losses`**：补齐分类必需的 **softmax 与交叉熵损失**（重点讲数值稳定性与那个漂亮的 `softmax - onehot` 梯度），为训练 MNIST 做最后准备。